In [ ]:
!pip install -q datasets transformers evaluate accelerate
!pip install -q "ray[tune]" scipy sklearn torch

In [ ]:
import time
import matplotlib as plt
import numpy as np
import torch
import torch.nn as nn
from torch import Tensor
from tqdm.notebook import tqdm  # Progress bar

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import evaluate

In [ ]:
'''Set-up GPU for use'''

gpu_avail = torch.cuda.is_available()
print(f"Is the GPU available? {gpu_avail}")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device", device)

# GPU operations have a separate seed
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

# Some operations on a GPU are implemented stochastic for efficiency
# Ensure that all operations are deterministic on GPU for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



Is the GPU available? False
Device cpu


# Load Data

In [ ]:
# Load preprocessed dataset
ucc = load_dataset('csv', data_files={'train': '/content/drive/My Drive/Colab Notebooks/ucc_train_no_scores.csv', 'test': '/content/drive/My Drive/Colab Notebooks/ucc_test_no_scores.csv'})


In [ ]:
classes = [class_ for class_ in list(ucc['train'].features)[1:] if class_]
class2id = {class_:id for id, class_ in enumerate(classes)}
id2class = {id:class_ for class_, id in class2id.items()}

print(ucc['train'][0])
print(classes)
print(class2id)
print(id2class)

{'comment': "The LEFT struggle with the TRUTH all the time. Furthermore, the 'D' in Democrat tells them to Deny, Deny, Deny, Distort and Destroy so it is very difficult for that type of Direction from the Democrats to be Disobeyed by their puppets.", 'antagonize': 1, 'condescending': 0, 'dismissive': 0, 'hostile': 1, 'sarcastic': 0}
['antagonize', 'condescending', 'dismissive', 'hostile', 'sarcastic']
{'antagonize': 0, 'condescending': 1, 'dismissive': 2, 'hostile': 3, 'sarcastic': 4}
{0: 'antagonize', 1: 'condescending', 2: 'dismissive', 3: 'hostile', 4: 'sarcastic'}
{'comment': "The LEFT struggle with the TRUTH all the time. Furthermore, the 'D' in Democrat tells them to Deny, Deny, Deny, Distort and Destroy so it is very difficult for that type of Direction from the Democrats to be Disobeyed by their puppets.", 'antagonize': 1, 'condescending': 0, 'dismissive': 0, 'hostile': 1, 'sarcastic': 0}


# Tokenize data

In [ ]:
# Define Tokenizer
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small', use_fast=True)

#tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')
#tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
#tokenizer = AutoTokenizer.from_pretrained('bert-large-uncased', use_fast=True)

/usr/local/lib/python3.10/dist-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:

def preprocess_function(example):

  # Gather list of positive labels
  all_labels = []
  for class_ in classes:
    if example[class_] == 1:
      all_labels.append(class_)

  # Convert labels to one-hot vector
  labels = [0.0 for i in range(len(classes))]
  for label in all_labels:
    label_id = class2id[label]
    labels[label_id] = 1

  example = tokenizer(example['comment'], truncation=True)
  example['labels'] = labels

  return example

# Tokenize data
tokenized_ucc = ucc.map(preprocess_function)

# Delete label columns now that we represent the labels as one-hot vec
tokenized_ucc = tokenized_ucc.remove_columns(classes)

# Split into train and test tokenized datasets
ucc_train_dataset = tokenized_ucc['train'].shuffle(seed=6)
ucc_eval_dataset = tokenized_ucc['test'].shuffle(seed=6)
# add '.select(range(1000))' at end to create smaller subset of dataset

Map:   0%|          | 0/2872 [00:00<?, ? examples/s]

Map:   0%|          | 0/4425 [00:00<?, ? examples/s]

In [ ]:
ucc_train_dataset[0]

{'comment': 'Better fix your (tinfoil) hat',
 'input_ids': [1, 6269, 2760, 290, 287, 297, 547, 46557, 285, 4751, 2],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [1.0, 1.0, 1.0, 0.0, 0.0]}

# Fine-tuning Pretrained Model

In [ ]:
def sigmoid(x):
  return 1/(1 + np.exp(-x))

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = sigmoid(predictions)  # concerned that sigmoid is not needed here and is squashing probabilities
    predictions = (predictions > 0.5).astype(int).reshape(-1)
    return accuracy.compute(predictions=predictions, references=labels.astype(int).reshape(-1))

In [ ]:
# Create a Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Include a training metric
accuracy = evaluate.load('accuracy')

# Define Pretrained Model
model = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-small',
                                                           num_labels=len(classes),
                                                           id2label=id2class, label2id=class2id,
                                                           problem_type='multi_label_classification')

#model = AutoModelForSequenceClassification.from_pretrained('distilbert/distilbert-base-uncased', num_labels=5)
#model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english', num_labels=5)
#model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-cased", num_labels=5)
#RoBerta - more robust than BERT
# XLNet - less params, also suitable for text classification


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Hyperparameter tuning

training_args = TrainingArguments(output_dir='deberta_ucc_tuned',
                                  evaluation_strategy='epoch',
                                  save_strategy='epoch',
                                  load_best_model_at_end=True,
                                  num_train_epochs=1)

#training_args = TrainingArguments(
#output_dir="my_awesome_model",
#learning_rate=2e-5,
#per_device_train_batch_size=3,
#per_device_eval_batch_size=3,
#num_train_epochs=2,
#weight_decay=0.01,
#evaluation_strategy="epoch",
#save_strategy="epoch",
#load_best_model_at_end=True,
#)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ucc_train_dataset,
    eval_dataset=ucc_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [ ]:
# test for inference:
text = 'This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three.'
inputs = tokenizer(text, return_tensors='pt')
print(inputs)
with torch.no_grad():
    logits = model(**inputs).logits
    losses = model(**inputs)
    print(logits)
    print(losses)
    print(losses.keys())

{'input_ids': tensor([[    1,   329,   284,   266, 14651,   260,   951,  1298,  9141,   264,
           262,  1116,   261,   304, 55700,   292,  1547,   264,   513,   260,
         16000,   282,   312,  1237,   265,   262,   475,   260,     2]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]])}
tensor([[-0.0160, -0.1003, -0.0372,  0.3685, -0.2123]])
SequenceClassifierOutput(loss=None, logits=tensor([[-0.0160, -0.1003, -0.0372,  0.3685, -0.2123]]), hidden_states=None, attentions=None)
odict_keys(['logits'])


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.539346,0.710508


TrainOutput(global_step=359, training_loss=0.6378820732775505, metrics={'train_runtime': 2522.9622, 'train_samples_per_second': 1.138, 'train_steps_per_second': 0.142, 'total_flos': 37108667131536.0, 'train_loss': 0.6378820732775505, 'epoch': 1.0})

# Inference

In [ ]:
# Input tokenize data into pre-trained model to obtain an embedding

In [ ]:
# Use obtained embeddings as features for classification network

In [ ]:
dataset = load_dataset('knowledgator/events_classification_biotech')

his_classes = [class_ for class_ in dataset['train'].features['label 1'].names if class_]
his_class2id = {class_:id for id, class_ in enumerate(his_classes)}
his_id2class = {id:class_ for class_, id in his_class2id.items()}

print(his_classes)
print(his_class2id)
print(his_id2class)
print(dataset["train"][0], '\n')

def his_preprocess_function(example):
   text = f"{example['title']}.\n{example['content']}"
   all_labels = example['all_labels']
   labels = [0. for i in range(len(his_classes))]
   for label in all_labels:
       label_id = his_class2id[label]
       labels[label_id] = 1.

   example = tokenizer(text, truncation=True)
   example['labels'] = labels
   return example

tokenized_dataset = dataset.map(his_preprocess_function)
print(tokenized_dataset['train'][0])

/usr/local/lib/python3.10/dist-packages/datasets/load.py:1461: FutureWarning: The repository for knowledgator/events_classification_biotech contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/knowledgator/events_classification_biotech
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


['event organization', 'executive statement', 'regulatory approval', 'hiring', 'foundation', 'closing', 'partnerships & alliances', 'expanding industry', 'new initiatives or programs', 'm&a', 'service & product providing', 'event organisation', 'new initiatives & programs', 'subsidiary establishment', 'product launching & presentation', 'product updates', 'executive appointment', 'alliance & partnership', 'ipo exit', 'article publication', 'clinical trial sponsorship', 'company description', 'investment in public company', 'other', 'expanding geography', 'participation in an event', 'support & philanthropy', 'department establishment', 'funding round', 'patent publication']
{'event organization': 0, 'executive statement': 1, 'regulatory approval': 2, 'hiring': 3, 'foundation': 4, 'closing': 5, 'partnerships & alliances': 6, 'expanding industry': 7, 'new initiatives or programs': 8, 'm&a': 9, 'service & product providing': 10, 'event organisation': 11, 'new initiatives & programs': 12, 

Map:   0%|          | 0/2759 [00:00<?, ? examples/s]

Map:   0%|          | 0/381 [00:00<?, ? examples/s]

{'title': "Sarah Polley's Book Recommendations", 'content': 'Drive Your Plow Over the Bones of The Dead\nby Olga Tokarczuk. I am an incredibly slow reader, but the tone and specificity of the world she creates in this book was something I couldnt leave behind until it was done. Also: All We Sawby Anne Michaels, Fight Nightby Miriam Toews, and The Summer Before the Darkby Doris Lessing.\nId like turned into a Netflix show:\nby Amia Srinivasan. One of the most brain-shattering books Ive ever read. Her thinking is so electrically rigorous and fearless. (I double DARE them to make this into a Netflix show!)\n...I last bought:\n. I rediscovered her poetry lately, and I feel like I dont want to read anything else for a while. She owns desire and submerged things.\n...has the greatest ending:\nby J.D. Salinger. The last page always leaves me breathless. The intimacy and truth of that final page is so arresting and almost painful to read.\nshould be on every college syllabus:\nby Anton Piatigo

In [ ]:
# Train

# Now, we can write a small training function. Remember our five steps: load a batch, obtain
# the predictions, calculate the loss, backpropagate, and update. Additionally, we have to push
# all data and model parameters to the device of our choice (GPU if available).

# Push model to device. Has to be only done once
model.to(device)

In [ ]:
# if didn't have pretrained model, would have to pretrain input data that also contains these preprocessing steps:

# Stemming? optional - can't use with tagging
# Tagging? tag each word represented by a token with its part of speech - can't use with stemming
# Lemmatization etc.